<a href="https://colab.research.google.com/github/tushar1298/nucligs_db/blob/main/NucLigs_SMILES_to_3D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title Generate 3D structures from resulted file
# ==========================================================
# INSTALL
# ==========================================================
!pip -q install rdkit openpyxl ipywidgets

# ==========================================================
# IMPORTS
# ==========================================================
import os
import io
import re
import shutil
import zipfile
import pandas as pd

from rdkit import Chem
from rdkit.Chem import AllChem

from google.colab import files
from IPython.display import display, clear_output
import ipywidgets as widgets

# ==========================================================
# OUTPUT DIRECTORY
# ==========================================================
OUTPUT_DIR = "Generated_PDBs"


def reset_output():

    if os.path.exists(OUTPUT_DIR):
        shutil.rmtree(OUTPUT_DIR)

    os.makedirs(
        OUTPUT_DIR,
        exist_ok=True
    )


reset_output()

# ==========================================================
# CLEAN FILENAMES
# ==========================================================
def clean_filename(text):

    text = str(text)

    text = re.sub(
        r'[\\/*?:"<>|]',
        "",
        text
    )

    text = text.strip()

    text = text.replace(
        " ",
        "_"
    )

    return text


# ==========================================================
# GENERATE PDB
# ==========================================================
def generate_pdb(
    smiles,
    compound_name
):

    try:

        mol = Chem.MolFromSmiles(
            smiles
        )

        if mol is None:
            return "Invalid SMILES"

        mol = Chem.AddHs(mol)

        params = AllChem.ETKDGv3()
        params.randomSeed = 42
        params.useRandomCoords = True

        status = AllChem.EmbedMolecule(
            mol,
            params
        )

        if status != 0:
            return "Embedding Failed"

        try:

            props = (
                AllChem
                .MMFFGetMoleculeProperties(
                    mol
                )
            )

            if props:
                AllChem.MMFFOptimizeMolecule(
                    mol
                )
            else:
                AllChem.UFFOptimizeMolecule(
                    mol
                )

        except:

            try:
                AllChem.UFFOptimizeMolecule(
                    mol
                )
            except:
                pass

        filename = (
            clean_filename(
                compound_name
            )
            + ".pdb"
        )

        path = os.path.join(
            OUTPUT_DIR,
            filename
        )

        Chem.MolToPDBFile(
            mol,
            path
        )

        if os.path.exists(path):
            return "Success"

        return "Save Failed"

    except Exception as e:
        return str(e)


# ==========================================================
# PROCESS COMPOUNDS
# ==========================================================
def process_compounds(
    compound_data,
    output_box
):

    with output_box:

        clear_output()

        reset_output()

        if len(compound_data) == 0:

            print(
                "No compounds found."
            )

            return

        results = []
        pdb_files = []

        print(
            f"Generating "
            f"{len(compound_data)} "
            f"3D structures...\n"
        )

        # --------------------------------------
        # GENERATE STRUCTURES
        # --------------------------------------
        for i, (
            name,
            smiles
        ) in enumerate(
            compound_data
        ):

            print(
                f"[{i+1}/"
                f"{len(compound_data)}] "
                f"{name}"
            )

            status = generate_pdb(
                smiles,
                name
            )

            results.append({
                "Name": name,
                "SMILES": smiles,
                "Status": status
            })

            if status == "Success":

                pdb_file = os.path.join(
                    OUTPUT_DIR,
                    clean_filename(
                        name
                    )
                    + ".pdb"
                )

                pdb_files.append(
                    pdb_file
                )

        # --------------------------------------
        # RESULTS DATAFRAME
        # --------------------------------------
        results_df = pd.DataFrame(
            results
        )

        print("\nSummary")
        display(results_df)

        # --------------------------------------
        # SAVE REPORT INSIDE OUTPUT FOLDER
        # --------------------------------------
        report_path = os.path.join(
            OUTPUT_DIR,
            "generation_report.csv"
        )

        results_df.to_csv(
            report_path,
            index=False
        )

        # --------------------------------------
        # CREATE SINGLE ZIP
        # --------------------------------------
        zip_name = (
            "Generated_PDBs.zip"
        )

        with zipfile.ZipFile(
            zip_name,
            "w",
            zipfile.ZIP_DEFLATED
        ) as zipf:

            # Add all PDB files
            for file in pdb_files:

                zipf.write(
                    file,
                    arcname=os.path.basename(
                        file
                    )
                )

            # Add report CSV
            zipf.write(
                report_path,
                arcname=
                "generation_report.csv"
            )

        print(
            f"\nGenerated "
            f"{len(pdb_files)} "
            f"PDB files."
        )

        print(
            "generation_report.csv "
            "included in ZIP."
        )

        # --------------------------------------
        # DOWNLOAD ONLY ZIP
        # --------------------------------------
        files.download(
            zip_name
        )

# ==========================================================
# TAB 1 : PASTE SMILES
# ==========================================================
paste_text = widgets.Textarea(
    placeholder=
"""Aspirin,CC(=O)OC1=CC=CC=C1C(=O)O
Caffeine,CN1C=NC2=C1C(=O)N(C(=O)N2C)C

OR

CCO
CCN
""",
    layout=widgets.Layout(
        width="100%",
        height="350px"
    )
)

paste_button = widgets.Button(
    description=
    "Generate PDBs",
    button_style="success"
)

paste_output = widgets.Output()


def paste_run(b):

    text = (
        paste_text
        .value
        .strip()
    )

    compounds = []

    for idx, line in enumerate(
        text.split("\n")
    ):

        line = line.strip()

        if not line:
            continue

        if "," in line:

            name, smiles = (
                line.split(
                    ",",
                    1
                )
            )

        else:

            smiles = line

            name = (
                f"Compound_"
                f"{idx+1}"
            )

        compounds.append(
            (
                name.strip(),
                smiles.strip()
            )
        )

    process_compounds(
        compounds,
        paste_output
    )


paste_button.on_click(
    paste_run
)

# ==========================================================
# TAB 2 : FILE UPLOAD
# ==========================================================
csv_output = widgets.Output()

csv_upload = widgets.FileUpload(
    accept=".csv,.xlsx,.xls",
    multiple=False
)

csv_button = widgets.Button(
    description=
    "Generate from File",
    button_style="info"
)

delete_button = widgets.Button(
    description=
    "Delete File",
    button_style="danger"
)


def run_csv(b):

    with csv_output:
        clear_output()

    if not csv_upload.value:

        with csv_output:
            print(
                "Upload file first."
            )

        return

    try:

        uploaded = list(
            csv_upload.value
        )[0]

        if isinstance(
            uploaded,
            dict
        ):

            content = uploaded.get(
                "content"
            )

        else:

            uploaded = list(
                csv_upload.value.values()
            )[0]

            content = uploaded[
                "content"
            ]

        # --------------------------------------
        # READ FILE
        # --------------------------------------
        df = None

        encodings = [
            "utf-8",
            "utf-8-sig",
            "latin1",
            "cp1252",
            "ISO-8859-1"
        ]

        for enc in encodings:

            try:

                df = pd.read_csv(
                    io.BytesIO(
                        content
                    ),
                    encoding=enc
                )

                break

            except:
                pass

        if df is None:

            try:

                df = pd.read_excel(
                    io.BytesIO(
                        content
                    )
                )

            except:

                with csv_output:
                    print(
                        "Could not "
                        "read file."
                    )

                return

        # --------------------------------------
        # CLEAN HEADERS
        # --------------------------------------
        df.columns = (
            df.columns
            .astype(str)
            .str.lower()
            .str.strip()
        )

        smiles_col = None
        nl_col = None
        name_col = None

        # SMILES
        for col in df.columns:

            if any(
                key in col
                for key in [
                    "smiles",
                    "smile",
                    "canonical_smiles",
                    "isomeric_smiles"
                ]
            ):

                smiles_col = col
                break

        # NL
        for col in df.columns:

            if any(
                key in col
                for key in [
                    "nl",
                    "identifier",
                    "compound id",
                    "id"
                ]
            ):

                nl_col = col
                break

        # NAME
        priority = [

            "trade name",
            "iupac name",
            "chemical name",
            "compound name",
            "molecule name",
            "name",
            "title"
        ]

        for p in priority:

            for col in df.columns:

                if (
                    p in col
                    and
                    "smiles"
                    not in col
                ):

                    name_col = col
                    break

            if name_col:
                break

        with csv_output:

            print(
                "Detected columns:"
            )

            print(
                list(df.columns)
            )

            print(
                f"\nNL: {nl_col}"
            )

            print(
                f"Name: {name_col}"
            )

            print(
                f"SMILES: {smiles_col}"
            )

        if smiles_col is None:

            with csv_output:
                print(
                    "\nSMILES column "
                    "not found."
                )

            return

        if (
            nl_col is None
            and
            name_col is None
        ):

            with csv_output:
                print(
                    "\nName/NL column "
                    "not found."
                )

            return

        required = [
            smiles_col
        ]

        if nl_col:
            required.append(
                nl_col
            )

        if name_col:
            required.append(
                name_col
            )

        df = df.dropna(
            subset=required
        )

        compounds = []

        for _, row in df.iterrows():

            smiles = str(
                row[smiles_col]
            ).strip()

            if (
                nl_col
                and
                name_col
            ):

                compound_name = (
                    f"{str(row[nl_col]).strip()}"
                    f"_"
                    f"{str(row[name_col]).strip()}"
                )

            elif nl_col:

                compound_name = str(
                    row[nl_col]
                ).strip()

            else:

                compound_name = str(
                    row[name_col]
                ).strip()

            compounds.append(
                (
                    compound_name,
                    smiles
                )
            )

        process_compounds(
            compounds,
            csv_output
        )

    except Exception as e:

        with csv_output:
            print(str(e))


def delete_file(b):

    global csv_upload

    with csv_output:

        clear_output()

        csv_upload.close()

        csv_upload = widgets.FileUpload(
            accept=".csv,.xlsx,.xls",
            multiple=False
        )

        csv_area.children = [

            csv_upload,

            widgets.HBox([
                csv_button,
                delete_button
            ]),

            csv_output
        ]

        print(
            "Uploaded file deleted."
        )


csv_button.on_click(
    run_csv
)

delete_button.on_click(
    delete_file
)

csv_area = widgets.VBox([
    csv_upload,
    widgets.HBox([
        csv_button,
        delete_button
    ]),
    csv_output
])

# ==========================================================
# DISPLAY TABS
# ==========================================================
tab1 = widgets.VBox([
    paste_text,
    paste_button,
    paste_output
])

tab2 = widgets.VBox([
    widgets.HTML(
        "<h3>Upload CSV/XLSX</h3>"
    ),
    widgets.HTML(
        """
        Automatically detects
        Name/NL and SMILES columns.
        Ignores other columns.
        """
    ),
    csv_area
])

tabs = widgets.Tab()

tabs.children = [
    tab1,
    tab2
]

tabs.set_title(
    0,
    "Paste SMILES"
)

tabs.set_title(
    1,
    "Upload File"
)

display(tabs)